# Replication script for the paper "Generating Abstract Tests for Asmeta Models by Using Code Transformations and EvoSuite"

This script is used to replicate the results of the analyses presented in the paper.

In [1]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
from IPython.display import display

DATA_FILE = Path('data.csv')
MODEL_LIST_FILE = Path('model_list.txt')
MODEL_LISTS_DIR = Path('model_lists')
VALID_MODEL_LIST_FILE = MODEL_LISTS_DIR / 'model_list_all_valid.txt'
IMAGE_DIR = Path('images')
VALID_STATUSES = {'OK', 'MUTATION_ERR'}
EVO_CONFIGURATIONS = ['evoavalla_budget_1', 'evoavalla_budget_5', 'evoavalla_budget_10']
INITIAL_CONFIGURATIONS = ['random', 'atgt'] + EVO_CONFIGURATIONS
MATCHED_RANDOM_CONFIGURATIONS = ['random4atgt'] + ['random4' + a for a in EVO_CONFIGURATIONS]
RANDOM_CONFIGURATIONS = [INITIAL_CONFIGURATIONS[0]] + MATCHED_RANDOM_CONFIGURATIONS
ALL_CONFIGURATIONS = INITIAL_CONFIGURATIONS + MATCHED_RANDOM_CONFIGURATIONS

def model_key(path):
    path = str(path).strip().replace('\\', '/')
    if 'asm_examples/' in path:
        path = path.split('asm_examples/', 1)[1]
    return path.removeprefix('./')

# Ignore blank lines and comments; preserve the order and spelling of model_list.txt.
model_list = [
    line.split('//', 1)[0].strip()
    for line in MODEL_LIST_FILE.read_text(encoding='utf-8-sig').splitlines()
    if line.split('//', 1)[0].strip()
]
model_keys = list(dict.fromkeys(model_key(model) for model in model_list))
df = pd.read_csv(DATA_FILE)
# The CSV calls its configuration field 'approach'; use distinct names in this notebook.
df = df.rename(columns={'approach': 'configuration'})
df['model_spec'] = df['asm_path'].map(model_key)
df['is_valid'] = df['status'].isin(VALID_STATUSES)

# Each configuration/specification pair has exactly one CSV record.
# Records outside model_list.txt are excluded from RQ1, RQ3, RQ4, plots, statistical tests,
# and model_list_all_valid.txt. Only the 'Other statuses in the CSV' diagnostics use the full CSV.
study_df = df[df['model_spec'].isin(model_keys) & df['configuration'].isin(ALL_CONFIGURATIONS)].copy()
validity = (
    study_df.pivot(index='model_spec', columns='configuration', values='is_valid')
    .reindex(index=model_keys, columns=ALL_CONFIGURATIONS)
    .fillna(False).astype(bool)
)
print(f'Specifications in model_list.txt: {len(model_keys)}')
outside_model_list = df.loc[~df['model_spec'].isin(model_keys)]
print(f'CSV records outside model_list.txt: {len(outside_model_list)}')
print(f'Distinct specifications outside model_list.txt: {outside_model_list["model_spec"].nunique()}')
for specification in sorted(outside_model_list['model_spec'].unique()):
    print(f'  {specification}')
missing = [a for a in ALL_CONFIGURATIONS if a not in set(study_df['configuration'])]
print(f'Configurations not yet present: {missing}')


Specifications in model_list.txt: 295
CSV records outside model_list.txt: 64
Distinct specifications outside model_list.txt: 13
  examples/GilbreathCardTrick/GilbreathCardTrickForAsmetaSMV_flat.asm
  examples/GilbreathCardTrick/GilbreathCardTrick_flat.asm
  examples/RushHour/RushHour_flat.asm
  examples/coffeeVendingMachine/coffeeVendingMachine_flat.asm
  examples/conwayGameOfLife/gameOfLifeAgents_flat.asm
  examples/conwayGameOfLife/gameOfLife_flat.asm
  examples/hemodialysisDevice/SCP2017/HemodialysisRef3_flat.asm
  examples/landingGearSystem/LGS_3L_flat.asm
  examples/petriNets/forAsmetaSMV/petriNet_forNuSMV_flat.asm
  examples/philosophers/philosophers3_flat.asm
  examples/roulette/roulette_flat.asm
  examples/simple_example/AdvancedClockDeciSecond.asm
  examples/ticTacToe/ticTacToe_simulator_flat.asm
Configurations not yet present: ['random4atgt', 'random4evoavalla_budget_1', 'random4evoavalla_budget_5', 'random4evoavalla_budget_10']


C:\Users\nicop\AppData\Local\Temp\ipykernel_13752\4152728284.py:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False).astype(bool)


## Check initial configuration records

Check that every active specification in `model_list.txt` has a CSV record for each of the five initial configurations. This checks record presence regardless of generation status: an unsuccessful generation still counts as a present record. Report missing configuration/specification pairs; the analysis can continue with missing records treated as invalid.

In [2]:
expected_pairs = pd.MultiIndex.from_product(
    [INITIAL_CONFIGURATIONS, model_keys], names=['configuration', 'model_spec']
)
present_pairs = pd.MultiIndex.from_frame(
    study_df.loc[study_df['configuration'].isin(INITIAL_CONFIGURATIONS),
                 ['configuration', 'model_spec']]
)
missing_pairs = expected_pairs.difference(present_pairs, sort=False).to_frame(index=False)
missing_counts = missing_pairs.groupby('configuration').size().reindex(
    INITIAL_CONFIGURATIONS, fill_value=0
)
record_check = pd.DataFrame({
    'expected_records': len(model_keys),
    'present_records': len(model_keys) - missing_counts,
    'missing_records': missing_counts,
})
display(record_check)
if missing_pairs.empty:
    print('PASS: every listed specification has a record for all five initial configurations.')
else:
    print(f'MISSING: {len(missing_pairs)} configuration/specification pairs across '
          f'{missing_pairs["model_spec"].nunique()} distinct specifications.')
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
        display(missing_pairs)


,expected_records,present_records,missing_records
configuration,,,
random,295,295,0
atgt,295,295,0
evoavalla_budget_1,295,295,0
evoavalla_budget_5,295,295,0
evoavalla_budget_10,295,295,0


PASS: every listed specification has a record for all five initial configurations.


## RQ1: Valid test generation

Count available and valid specifications for each of the five initial configurations. Each configuration/specification pair has exactly one CSV record. Missing records count as invalid. All RQ1 counts refer to active specifications in `model_list.txt`.

For every pair, report the inclusive intersection (`both_valid`: both configurations are valid, regardless of the others) and the exclusive intersection (`exclusive_intersection`: only these two configurations are valid; the other three are invalid or missing). After the pair table, report the total number of specifications valid for all five initial configurations.

In [3]:
initial_df = study_df[study_df['configuration'].isin(INITIAL_CONFIGURATIONS)]
rq1_summary = initial_df.groupby('configuration').agg(
    specifications=('configuration', 'size'),
    valid_specifications=('is_valid', 'sum'),
).reindex(INITIAL_CONFIGURATIONS, fill_value=0)
initial_validity = validity[INITIAL_CONFIGURATIONS]
display(rq1_summary)

initial_all_valid_models = initial_validity.index[initial_validity.all(axis=1)]

pair_counts = []
for left, right in combinations(INITIAL_CONFIGURATIONS, 2):
    both = initial_validity[left] & initial_validity[right]
    others = [c for c in INITIAL_CONFIGURATIONS if c not in (left, right)]
    pair_counts.append({
        'configuration_1': left, 'configuration_2': right,
        'both_valid': int(both.sum()),
        'exclusive_intersection': int((both & ~initial_validity[others].any(axis=1)).sum()),
    })
display(pd.DataFrame(pair_counts))

print(f'Valid for all {len(INITIAL_CONFIGURATIONS)} initial configurations: {len(initial_all_valid_models)}')


,specifications,valid_specifications
configuration,,
random,295,174
atgt,295,51
evoavalla_budget_1,295,102
evoavalla_budget_5,295,107
evoavalla_budget_10,295,110


,configuration_1,configuration_2,both_valid,exclusive_intersection
0,random,atgt,50,3
1,random,evoavalla_budget_1,93,0
2,random,evoavalla_budget_5,98,1
3,random,evoavalla_budget_10,100,3
4,atgt,evoavalla_budget_1,48,0
5,atgt,evoavalla_budget_5,48,0
6,atgt,evoavalla_budget_10,48,0
7,evoavalla_budget_1,evoavalla_budget_5,100,0
8,evoavalla_budget_1,evoavalla_budget_10,101,1
9,evoavalla_budget_5,evoavalla_budget_10,105,1


Valid for all 5 initial configurations: 47


## Utility: Export valid specification lists

Create `model_lists/` and write `model_list_all_valid.txt` using the intersection of the five initial configurations from RQ1. Also write `model_list_<configuration>_valid.txt` for each available configuration, including the size-matched Random configurations when present. Keep specifications from `model_list.txt` in their original order and path format; valid statuses are `OK` and `MUTATION_ERR`. Skip unavailable configurations; skip the common-list export if an initial configuration is entirely missing.

In [4]:
MODEL_LISTS_DIR.mkdir(exist_ok=True)

def export_model_list(path, valid_models):
    valid_models = set(valid_models)
    models = [m for m in model_list if model_key(m) in valid_models]
    path.write_text('\n'.join(models) + ('\n' if models else ''), encoding='utf-8')
    print(f'Wrote {len(models)} specifications to {path}')
    return models

missing_initial = [a for a in INITIAL_CONFIGURATIONS if a not in set(initial_df['configuration'])]
valid_model_list = []
if missing_initial:
    print(f'Common-list export skipped; missing initial configurations: {missing_initial}')
else:
    valid_model_list = export_model_list(VALID_MODEL_LIST_FILE, initial_all_valid_models)

for configuration in ALL_CONFIGURATIONS:
    if configuration not in set(study_df['configuration']):
        print(f'Configuration-list export skipped: {configuration} has no records.')
        continue
    valid_models = validity.index[validity[configuration]]
    export_model_list(
        MODEL_LISTS_DIR / f'model_list_{configuration}_valid.txt', valid_models
    )


Wrote 47 specifications to model_lists\model_list_all_valid.txt
Wrote 174 specifications to model_lists\model_list_random_valid.txt
Wrote 51 specifications to model_lists\model_list_atgt_valid.txt
Wrote 102 specifications to model_lists\model_list_evoavalla_budget_1_valid.txt
Wrote 107 specifications to model_lists\model_list_evoavalla_budget_5_valid.txt
Wrote 110 specifications to model_lists\model_list_evoavalla_budget_10_valid.txt
Configuration-list export skipped: random4atgt has no records.
Configuration-list export skipped: random4evoavalla_budget_1 has no records.
Configuration-list export skipped: random4evoavalla_budget_5 has no records.
Configuration-list export skipped: random4evoavalla_budget_10 has no records.


## Other statuses in the CSV

Report all records whose status is neither `OK` nor `NO_VALID_SCENARIOS`, including `MUTATION_ERR` (which still counts as a valid generation). These diagnostics cover the entire CSV, including records outside the study list, and identify each specification.

In [5]:
other_statuses = df[~df['status'].isin({'OK', 'NO_VALID_SCENARIOS'})]
print(f'Total records with other statuses: {len(other_statuses)}')
display(other_statuses.groupby(['status', 'configuration'], dropna=False).size().rename('records').to_frame())
display(other_statuses.groupby(
    ['status', 'configuration', 'model_spec'], dropna=False
).size().rename('records').to_frame())


Total records with other statuses: 9


records
status       configuration               
MUTATION_ERR evoavalla_budget_1         2
             evoavalla_budget_10        2
             evoavalla_budget_5         2
             random                     3

records
status       configuration       model_spec                                                 
MUTATION_ERR evoavalla_budget_1  examples/filosofiEciliegie/filosofiEciliegieAge...        1
                                 examples/filosofiEciliegie/filosofiEciliegieAge...        1
             evoavalla_budget_10 examples/filosofiEciliegie/filosofiEciliegieAge...        1
                                 examples/filosofiEciliegie/filosofiEciliegieAge...        1
             evoavalla_budget_5  examples/filosofiEciliegie/filosofiEciliegieAge...        1
                                 examples/filosofiEciliegie/filosofiEciliegieAge...        1
             random              examples/fsmsemantics/meta_hooking/FSM.asm                1
                                 examples/fsmsemantics/meta_hooking/myFSM.asm              1
                                 examples/models/ordersystem.asm                           1

## Common data for RQ3 and RQ4

Require valid results for all nine configurations. Pool the three EvoAvalla budgets into EvoAvalla, and all five Random configurations (the initial baseline and the four `random4...` configurations) into Random.

Boxplots and descriptive statistics use all pooled observations. Statistical tests compare separate configurations, paired by specification, without averaging budgets or repeating ATGT observations within a test. The two families are RQ3 (6 tests) and RQ4 (115 tests); Holm correction at 0.05 is applied separately to each family. This controls errors within each RQ, not across the entire paper.

In [6]:
common_valid_models = validity.index[validity.all(axis=1)]
common_df = study_df[
    study_df['model_spec'].isin(common_valid_models) & study_df['is_valid']
].copy()
analysis_ready = not common_df.empty
print(f'Specifications valid for all nine configurations: {len(common_valid_models)}')
if not analysis_ready:
    print('RQ3/RQ4 skipped: no common valid specifications yet.')

COVERAGE_TYPES = ['branch_coverage', 'rule_coverage', 'update_rule_coverage', 'macro_coverage']
MUTATION_COLUMNS = [
    'casemutator_score', 'ruleremover_score', 'chooserulemutator_score',
    'condnegator_score', 'forallmutator_score', 'partoseqmutator_score',
    'seqtoparmutator_score', 'condremover_score',
]
QUALITY_METRICS = COVERAGE_TYPES + ['mutation_score']
SIZE_METRICS = ['n_scenarios', 'n_step']
group_map = {
    **{a: 'EvoAvalla' for a in EVO_CONFIGURATIONS}, 'atgt': 'ATGT',
    **{a: 'Random' for a in RANDOM_CONFIGURATIONS},
}
if analysis_ready:
    numeric_columns = SIZE_METRICS + COVERAGE_TYPES + MUTATION_COLUMNS
    common_df[numeric_columns] = common_df[numeric_columns].apply(pd.to_numeric, errors='raise')
    # All-missing mutation scores stay NaN (including failed mutation analyses).
    common_df['mutation_score'] = common_df[MUTATION_COLUMNS].mean(axis=1, skipna=True)

common_df['approach'] = common_df['configuration'].map(group_map)
pooled_df = common_df[common_df['approach'].notna()].copy()


Specifications valid for all nine configurations: 0
RQ3/RQ4 skipped: no common valid specifications yet.


## Statistical analysis: How to read the results

For each configuration pair, compare values from the same specification using a two-sided Wilcoxon test. Apply Holm correction separately to RQ3 (6 tests) and RQ4 (115 tests), with a significance threshold of 0.05.

The results table shows:
- `pairs`: specifications with both metric values available.
- `p_value`: original Wilcoxon p-value.
- `p_holm`: corrected p-value; use this value to decide significance.
- `result`: "Significant difference" if p_holm <= 0.05, otherwise "Not significant".
- `median_difference`: median of the paired differences, left minus right.
- `median_direction`: which configuration has higher values according to that median. This is descriptive; it is not a separate statistical test.

A significant p-value indicates evidence of a difference; it does not by itself indicate which configuration is higher. Read the paired difference alongside it. A zero median does not identify a direction, even if the test is significant. A non-significant result does not establish equivalence. For coverage and mutation score, higher values are better; for suite size, larger or smaller does not by itself mean better.

If every observed difference is exactly zero, report "Identical observed values" and use p = 1 by convention without running Wilcoxon. If no complete pairs are available, report "Not testable". Missing metric values are omitted pairwise. Untestable comparisons remain in the planned Holm family as p = 1 for adjustment only.

Descriptive tables report count, mean, sample variance and standard deviation. Tests use separate configurations; only boxplots and descriptive approach summaries pool configurations. Wilcoxon assumes independent specification-level differences with a symmetric difference distribution.

In [7]:
sns.set_theme(style='whitegrid')
PLOT_ORDER = ['EvoAvalla', 'ATGT', 'Random']
PALETTE = {'EvoAvalla': '#ea96a3', 'ATGT': '#98a247', 'Random': '#4aaba4'}
ALPHA = 0.05

def descriptive_stats(data, group, metrics):
    return data.groupby(group)[metrics].agg(['count', 'mean', 'var', 'std'])

def configuration_pairs(left_name, right_name):
    columns = ['model_spec'] + SIZE_METRICS + QUALITY_METRICS
    return common_df.loc[common_df['configuration'] == left_name, columns].merge(
        common_df.loc[common_df['configuration'] == right_name, columns],
        on='model_spec', suffixes=('_left', '_right'), validate='one_to_one'
    )

def equal_size_mask(pairs):
    sizes = [m + suffix for m in SIZE_METRICS for suffix in ('_left', '_right')]
    return (
        pairs[sizes].notna().all(axis=1)
        & pairs['n_scenarios_left'].eq(pairs['n_scenarios_right'])
        & pairs['n_step_left'].eq(pairs['n_step_right'])
    )

def paired_test(pairs, metric, left_name, right_name):
    complete = pairs[[metric + '_left', metric + '_right']].dropna()
    differences = complete.iloc[:, 0] - complete.iloc[:, 1]
    median = differences.median()
    if complete.empty:
        direction = 'Not available'
    elif median > 0:
        direction = 'Left higher'
    elif median < 0:
        direction = 'Right higher'
    else:
        direction = 'Zero median difference'
    result = {
        'left': left_name, 'right': right_name, 'metric': metric,
        'pairs': len(complete), 'median_difference': median,
        'median_direction': direction,
    }
    if complete.empty:
        result.update(statistic=float('nan'), p_value=float('nan'), note='No complete pairs')
    elif (differences == 0).all():
        result.update(statistic=float('nan'), p_value=1.0, note='All differences are zero')
    else:
        test = wilcoxon(differences, alternative='two-sided', zero_method='wilcox', nan_policy='omit')
        result.update(statistic=test.statistic, p_value=test.pvalue, note='')
    return result

def holm_adjust(results, family, planned_tests):
    results = results.copy()
    if len(results) != planned_tests:
        raise ValueError(f'{family}: expected {planned_tests} planned tests, got {len(results)}')
    # Retain untestable hypotheses in the planned family, without declaring a result for them.
    ordered = results['p_value'].fillna(1.0).sort_values(kind='stable')
    adjusted = (ordered * list(range(planned_tests, 0, -1))).cummax().clip(upper=1.0)
    results['p_holm'] = adjusted.reindex(results.index).where(results['p_value'].notna())
    results['family'] = family
    results['family_size'] = planned_tests
    results['significant'] = results['p_holm'] <= ALPHA
    return results

def classify_result(row):
    if pd.isna(row['p_holm']):
        return 'Not testable'
    if row['note'] == 'All differences are zero':
        return 'Identical observed values'
    return 'Significant difference' if row['significant'] else 'Not significant'

def report_results(results, quality=False):
    print('Summary: significance after Holm correction.')
    summary = pd.crosstab(results['metric'], results['result']).reindex(
        columns=['Significant difference', 'Not significant',
                 'Identical observed values', 'Not testable'], fill_value=0
    )
    summary['comparisons'] = summary.sum(axis=1)
    display(summary)
    print('Configuration-level results: median_difference = left minus right.')
    columns = ['metric', 'left', 'right', 'pairs', 'statistic', 'p_value',
               'p_holm', 'result', 'median_difference', 'median_direction', 'note']
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(results[columns])
    for metric in results['metric'].unique():
        subset = results[results['metric'] == metric]
        significant = subset[subset['significant']]
        left_higher = int((significant['median_difference'] > 0).sum())
        right_higher = int((significant['median_difference'] < 0).sum())
        zero_median = int((significant['median_difference'] == 0).sum())
        print(f'{metric}: {len(significant)}/{len(subset)} comparisons have a significant difference.')
        print(f'Among these, the paired median is positive in {left_higher}, '
              f'negative in {right_higher}, and zero in {zero_median}.')
    if quality:
        print('Left is the expected higher configuration for ATGT > EvoAvalla > Random.')
        print('A significant difference with a positive paired median is consistent with that ordering; '
              'a negative median points in the opposite direction. '
              'Non-significant results do not establish an ordering.')
    else:
        print('Left is ATGT. A positive paired median means more tests/steps for ATGT; '
              'a negative median means fewer. Suite size alone does not measure effectiveness.')

def boxplot(data, metric, order, filename, quality=False):
    if data[metric].notna().sum() == 0:
        print(f'Plot skipped: no values for {metric}')
        return
    IMAGE_DIR.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(2.4, 3.2))
    # Explicit arrays also support older seaborn versions without legend=False.
    values = [data.loc[data['approach'] == approach, metric].dropna() for approach in order]
    boxes = ax.boxplot(values, patch_artist=True, showfliers=False)
    for box, approach in zip(boxes['boxes'], order):
        box.set_facecolor(PALETTE[approach])
    ax.set_xticks(range(1, len(order) + 1), order, rotation=90)
    ax.set_ylabel(metric.replace('_', ' ').capitalize())
    if quality:
        ax.set_ylim(0, 1.01)
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / filename, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## RQ3: Size of test suites

Compare ATGT with each of the three EvoAvalla budgets for `n_scenarios` and `n_step`. Apply Holm once to the six tests. Show significance and the median paired difference separately: positive means ATGT has more tests/steps, negative means fewer. Smaller suite size alone does not imply higher effectiveness.

Keep two boxes per plot: all three EvoAvalla configurations pooled, and ATGT. Report count, mean, sample variance and standard deviation both by approach and by configuration.

In [8]:
rq3_tests = pd.DataFrame()
if analysis_ready:
    size_df = pooled_df[pooled_df['approach'].isin(['EvoAvalla', 'ATGT'])]
    display(descriptive_stats(size_df, 'approach', SIZE_METRICS))
    display(descriptive_stats(size_df, 'configuration', SIZE_METRICS))
    results = []
    for evo in EVO_CONFIGURATIONS:
        pairs = configuration_pairs('atgt', evo)
        results.extend(paired_test(pairs, metric, 'atgt', evo) for metric in SIZE_METRICS)
    rq3_tests = holm_adjust(pd.DataFrame(results), 'RQ3', planned_tests=6)
    rq3_tests['result'] = rq3_tests.apply(classify_result, axis=1)
    report_results(rq3_tests)
    for metric in SIZE_METRICS:
        boxplot(size_df, metric, ['EvoAvalla', 'ATGT'], f'boxplot_{metric}.pdf')


## RQ4: Coverage and mutation score

Each figure has three boxes: EvoAvalla (three configurations), ATGT, and Random (all five configurations). Descriptive summaries suggest an ordering; statistical evidence is assessed separately for each configuration pair.

For each of the five metrics, test 23 comparisons: ATGT vs 3 EvoAvalla, ATGT vs 5 Random, and 3 EvoAvalla vs 5 Random. Apply Holm once to all 115 tests. The left configuration is always the expected higher-performing approach for the proposed ordering ATGT > EvoAvalla > Random. This is a hypothesis to assess, not an assumed result.

Show the number of significant differences per metric, followed by the configuration-level results. Read the median paired difference to describe the direction; the p-value alone does not confirm ATGT > EvoAvalla > Random. Conclusions are conditional on specifications valid for all nine configurations.

The four size-matched comparisons per metric belong to the same family. For these pairs, use only specifications with equal `n_scenarios` and `n_step` on both sides, and reuse these exact tests and adjusted p-values in the next section. Other comparisons use all common valid specifications. Missing metric values may further reduce the number of pairs.

In [9]:
rq4_tests = pd.DataFrame()
size_audit = []
size_mismatches = {}
if analysis_ready:
    display(descriptive_stats(pooled_df, 'approach', QUALITY_METRICS))
    display(descriptive_stats(pooled_df, 'configuration', QUALITY_METRICS))
    size_matched_pairs = [('atgt', 'random4atgt')] + [(a, 'random4' + a) for a in EVO_CONFIGURATIONS]
    comparisons = (
        [('atgt', a, 'ATGT vs EvoAvalla') for a in EVO_CONFIGURATIONS]
        + [('atgt', a, 'ATGT vs Random') for a in RANDOM_CONFIGURATIONS]
        + [(e, r, 'EvoAvalla vs Random') for e in EVO_CONFIGURATIONS for r in RANDOM_CONFIGURATIONS]
    )
    results = []
    for left_name, right_name, group in comparisons:
        pairs = configuration_pairs(left_name, right_name)
        same_size_comparison = (left_name, right_name) in size_matched_pairs
        if same_size_comparison:
            same_size = equal_size_mask(pairs)
            size_audit.append({
                'left': left_name, 'right': right_name, 'candidate_pairs': len(pairs),
                'equal_size_pairs': int(same_size.sum()),
                'excluded_size_mismatches': int((~same_size).sum()),
            })
            size_mismatches[(left_name, right_name)] = pairs.loc[~same_size].copy()
            pairs = pairs.loc[same_size].copy()
        for metric in QUALITY_METRICS:
            result = paired_test(pairs, metric, left_name, right_name)
            result.update(comparison_group=group, equal_size=same_size_comparison)
            results.append(result)
    rq4_tests = holm_adjust(pd.DataFrame(results), 'RQ4', planned_tests=115)
    rq4_tests['result'] = rq4_tests.apply(classify_result, axis=1)
    report_results(rq4_tests, quality=True)
    for metric in QUALITY_METRICS:
        boxplot(pooled_df, metric, PLOT_ORDER, f'boxplot_{metric}.pdf', quality=True)


## RQ4: Paired comparisons at equal suite size

Display the four configuration pairs separately: ATGT vs `random4atgt`, and each EvoAvalla budget vs its matching `random4evoavalla_budget_*`. Verify equality of both test count and step count, reporting excluded mismatches.

These 20 metric-level tests are a subset of the 115 RQ4 tests. Reuse their Holm-adjusted p-values; do not add another family or average budgets. Significant advantages for the non-random approaches support the interpretation that suite size alone does not explain coverage and mutation score. Non-significant differences do not establish equivalence.

The unit is a test suite for a specification/configuration; the summary CSV does not allow individual scenarios to be paired.

In [10]:
if analysis_ready:
    display(pd.DataFrame(size_audit))
    for (left_name, right_name), mismatches in size_mismatches.items():
        if not mismatches.empty:
            print(f'Size mismatches: {left_name} vs {right_name}')
            display(mismatches[['model_spec'] + [
                metric + suffix for metric in SIZE_METRICS for suffix in ('_left', '_right')
            ]])
    print('Equal-size results: Holm adjustment from the full 115-test RQ4 family.')
    equal_size_tests = rq4_tests[rq4_tests['equal_size']].copy()
    report_results(equal_size_tests, quality=True)
